# Study 925 — Front-End Trend

**If short-term interest rates have been falling for three months, should you go and buy
the long bond?**

Central banks do not change their minds at random. When the Federal Reserve starts
cutting, it usually keeps cutting for a year or more; when it starts hiking, likewise. So
here is a rule almost anyone can follow. Look at the yield on the 3-month Treasury bill
(`^IRX`). Compare it with where it stood three months ago.

- **Rates falling?** Own long-dated Treasuries (**TLT**) — falling rates push bond prices
  up, and the longest bonds move the most.
- **Rates rising or flat?** Sit in Treasury bills (**BIL**) and collect the short rate.

Check daily, act the next morning, pay 2 bps of your portfolio each time you switch.

The benchmark it has to beat is deliberately unglamorous: **just holding IEF**, the 7-10
year Treasury fund, from start to finish and never thinking about rates again.

*Real-tape numbers below are the frozen headline from [`docs/results.md`](../docs/results.md) — SHY/IEF/TLT/BIL total-return closes plus the `^IRX` yield, 2007-05-30 → 2026-06-30 (4,800 days), Fingerprint `c05691fe8719`, as-of 2026-06-30. Live cells run the offline synthetic control only, and are labelled as such.*


## 1. The scoreboard

Everything below is measured **above cash** — we subtract what Treasury bills themselves paid. That matters enormously here: for a third of this sample bills paid nearly nothing and for another stretch they paid 5%, and a rule that spends half its life in bills would otherwise get credit for the Fed's generosity rather than for its own cleverness.

> 🔬 **For the quants** — every arm is a daily simple excess return over BIL's own total return, so the cash leg cancels in every pairwise difference and the Sharpe race is like-for-like.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4800, 'fp': 'c05691fe8719', 'lookback': 63, 'cost_bps': 2.0, 'in_frac': 0.478, 'n_switches': 321, 'switches_per_year': 17, 'sw_sharpe': 0.154, 'sw_sharpe_gross': 0.185, 'sw_cagr': 1.09, 'sw_vol': 10.98, 'sw_dd': -28.2, 'sw_t': 0.73, 'ief_sharpe': 0.293, 'ief_cagr': 1.82, 'ief_vol': 6.99, 'ief_dd': -23.9, 'ief_t': 1.37, 'tlt_sharpe': 0.184, 'tlt_cagr': 1.66, 'tlt_vol': 15.24, 'tlt_dd': -48.4, 'tlt_t': 0.89, 'rnd_sharpe': -0.316, 'rnd_cagr': -3.74, 'rnd_dd': -47.2, 'adv': -0.139, 't_vs_ief': -0.2, 't_vs_tlt': -0.52, 'adv_one_seed': 0.469, 't_one_seed': 2.01, 'ci_sw_lo': -0.283, 'ci_sw_hi': 0.554, 'ci_sw_neg': 24.3, 'ci_ief_lo': -0.136, 'ci_ief_hi': 0.704, 'ci_ief_neg': 9.3, 'ci_diff_lo': -0.498, 'ci_diff_hi': 0.221, 'ci_diff_neg': 76.4, 'rnd_net_adv': 0.268, 'rnd_net_sd': 0.177, 'rnd_net_t': 1.2, 'rnd_net_share': 20, 'rnd_gross_adv': 0.06, 'rnd_gross_sd': 0.175, 'rnd_gross_t': 0.29, 'rnd_gross_share': 0, 'rnd_flips': 2400, 'blend_w': 47.8, 'blend_turnover': 91, 'blend_net_sharpe': 0.182, 'blend_net_adv': -0.028, 'blend_net_t': 0.23, 'blend_gross_sharpe': 0.184, 'blend_gross_adv': 0.001, 'blend_gross_t': 0.43, 'on_bp': 1.69, 'on_n': 2263, 'on_t': 0.87, 'off_bp': 0.59, 'off_n': 2473, 'off_t': 0.36, 'spread_bp': 1.09, 'spread_hac': 0.43, 'spread_welch': 0.39, 'era_e_n': 2100, 'era_e_ief': 0.73, 'era_e_sw': 0.28, 'era_e_adv': -0.452, 'era_e_t': -0.71, 'era_e_frac': 58, 'era_e_dd_ief': -10.4, 'era_e_dd_sw': -27.1, 'era_l_n': 2635, 'era_l_ief': -0.11, 'era_l_sw': 0.03, 'era_l_adv': 0.138, 'era_l_t': 0.44, 'era_l_frac': 39, 'era_l_dd_ief': -23.9, 'era_l_dd_sw': -28.2, 'lb21_adv': 0.056, 'lb21_t': 0.99, 'lb42_adv': -0.178, 'lb42_t': -0.41, 'lb63_adv': -0.139, 'lb63_t': -0.2, 'lb126_adv': -0.122, 'lb126_t': -0.11, 'lb252_adv': 0.083, 'lb252_t': 1.25, 'cost0_adv': -0.108, 'cost0_t': -0.01, 'cost10_adv': -0.264, 'cost10_t': -0.96, 'cost25_adv': -0.496, 'cost25_t': -2.35, 'years': ((2007, 6.7, 5.5, 7.3, 1.2, 99), (2008, 26.1, 17.9, 34.0, 8.2, 78), (2009, -21.5, -6.6, -21.8, -14.9, 67), (2010, -2.4, 9.4, 9.0, -11.7, 47), (2011, 26.4, 15.6, 34.0, 10.7, 73), (2012, 4.5, 3.7, 2.4, 0.8, 20), (2013, -14.8, -6.1, -13.4, -8.7, 59), (2014, 24.9, 9.1, 27.3, 15.9, 73), (2015, -11.3, 1.5, -1.8, -12.8, 38), (2016, 2.7, 1.0, 1.2, 1.7, 24), (2017, -1.4, 2.6, 9.2, -3.9, 2), (2018, 1.7, 1.0, -1.6, 0.7, 0), (2019, 6.9, 8.0, 14.1, -1.2, 67), (2020, 6.1, 10.0, 18.2, -3.9, 83), (2021, -15.2, -3.3, -4.6, -11.9, 56), (2022, 1.4, -15.2, -31.2, 16.5, 0), (2023, 21.0, 3.6, 2.8, 17.3, 15), (2024, -9.4, -0.6, -8.1, -8.8, 71), (2025, 8.8, 8.0, 4.2, 0.8, 72), (2026, 1.8, -0.1, 1.0, 1.9, 47)), 'yrs_beat': 11, 'yrs_total': 20, 'gap_mean': -0.1, 'gap_median': 0.76, 'y2022_sw': 1.4, 'y2022_ief': -15.2, 'y2022_tlt': -31.2, 'y2022_sessions': 0, 'y2022_total': 251, 'y2023_sw': 21.0, 'y2023_ief': 3.6, 'y2023_days': 37, 'y2023_tlt_window': 14.7, 'y2023_strat_window': 16.2, 'y2023_bil': 4.9, 'shy_sharpe': 0.331, 'shy_adv': 0.061, 'shy_t': 1.43, 'shy_frac': 85, 'shy_switches': 68, 'shy_gross_rnd': 0.172, 'shy_gross_rnd_t': 1.36, 'shy_blend_w': 85.2, 'shy_blend_sharpe': 0.177, 'shy_blend_adv': 0.153, 'shy_blend_t': 1.92, 'shy_blend_gross_adv': 0.158, 'shy_blend_gross_t': 1.97, 'shy_blend_early_adv': 0.005, 'shy_blend_early_t': 0.51, 'shy_blend_early_frac': 99, 'shy_blend_late_adv': 0.243, 'shy_blend_late_t': 1.52, 'shy_blend_ex22_adv': 0.156, 'shy_blend_ex22_t': 1.24, 'syn_blend_adv': 1.545, 'syn_blend_t': 4.45, 'syn_pl_adv': 1.78, 'syn_pl_t': 4.66, 'syn_pl_mean': 2.15, 'syn_pl_fire': 10, 'syn_nl_mean': -0.05, 'syn_nl_sd': 0.203, 'syn_nl_fire': 0}
rows = [('switch (the rule)', R['sw_sharpe'], R['sw_cagr'], R['sw_vol'], R['sw_dd']),
        ('static IEF',       R['ief_sharpe'], R['ief_cagr'], R['ief_vol'], R['ief_dd']),
        ('static TLT',       R['tlt_sharpe'], R['tlt_cagr'], R['tlt_vol'], R['tlt_dd']),
        ('random control',   R['rnd_sharpe'], R['rnd_cagr'], float('nan'), R['rnd_dd'])]
print(f"{'arm':18s}{'exSharpe':>10s}{'exCAGR':>9s}{'vol':>8s}{'worst loss':>12s}")
for name, sh, cg, vol, dd in rows:
    v = '   n/a' if vol != vol else f'{vol:6.2f}%'
    print(f'{name:18s}{sh:+10.3f}{cg:+8.2f}%{v:>8s}{dd:+11.1f}%')
print()
print(f"advantage of the rule over just holding IEF: {R['adv']:+.3f} Sharpe")
print(f"how sure are we?  HAC t = {R['t_vs_ief']:+.2f}   (the desk's bar is |t| >= 2)")

arm                 exSharpe   exCAGR     vol  worst loss
switch (the rule)     +0.154   +1.09%  10.98%      -28.2%
static IEF            +0.293   +1.82%   6.99%      -23.9%
static TLT            +0.184   +1.66%  15.24%      -48.4%
random control        -0.316   -3.74%     n/a      -47.2%

advantage of the rule over just holding IEF: -0.139 Sharpe
how sure are we?  HAC t = -0.20   (the desk's bar is |t| >= 2)


## 2. The rule is not just worse — it is worse in every way

A timing rule is allowed to earn less if it makes the ride smoother. This one does not. Against simply holding IEF it delivers a **lower** return above cash (1.09% vs 1.82%), **more** volatility (11.0% vs 7.0%) and a **deeper** worst loss (-28.2% vs -23.9%) — while asking you to trade about **17 times a year**. There is no dimension on which it wins.

## 3. So why does everyone remember this rule working?

Because of two spectacular years.

**2022.** Rates rose relentlessly all year. The rule was in Treasury bills for **251 of 251 sessions** — it never once owned duration — and finished **+1.4%**, while IEF lost **-15.2%** and TLT lost **-31.2%**. A 16-point win in the year that broke the 60/40 portfolio.

**2023.** The rule owned long bonds for just **37 sessions** — 6 November to 29 December — during which **TLT rose +14.7%**; the rule's own return over that stretch was +16.2% (it sat out the one down day inside the window). Bills paid 4.9% for the other eleven months. Result: **+21.0%** against IEF's +3.6%.

Those two years are genuine — not artefacts, not look-ahead. So here is the whole scorecard, every year, nothing left out.

In [2]:
print(f"{'year':>6s}{'switch':>10s}{'IEF':>10s}{'TLT':>10s}{'gap':>10s}{'in duration':>13s}")
for y, sw, ief, tlt, gap, frac in R['years']:
    print(f'{y:>6d}{sw:+9.1f}%{ief:+9.1f}%{tlt:+9.1f}%{gap:+9.1f}%{frac:>12d}%')
print()
print(f"beats IEF in {R['yrs_beat']} of {R['yrs_total']} years; "
      f"mean gap {R['gap_mean']:+.2f} pp/yr, median {R['gap_median']:+.2f} pp/yr")
print('a coin-toss hit rate with a slightly negative mean -- what no information looks like.')

  year    switch       IEF       TLT       gap  in duration
  2007     +6.7%     +5.5%     +7.3%     +1.2%          99%
  2008    +26.1%    +17.9%    +34.0%     +8.2%          78%
  2009    -21.5%     -6.6%    -21.8%    -14.9%          67%
  2010     -2.4%     +9.4%     +9.0%    -11.7%          47%
  2011    +26.4%    +15.6%    +34.0%    +10.7%          73%
  2012     +4.5%     +3.7%     +2.4%     +0.8%          20%
  2013    -14.8%     -6.1%    -13.4%     -8.7%          59%
  2014    +24.9%     +9.1%    +27.3%    +15.9%          73%
  2015    -11.3%     +1.5%     -1.8%    -12.8%          38%
  2016     +2.7%     +1.0%     +1.2%     +1.7%          24%
  2017     -1.4%     +2.6%     +9.2%     -3.9%           2%
  2018     +1.7%     +1.0%     -1.6%     +0.7%           0%
  2019     +6.9%     +8.0%    +14.1%     -1.2%          67%
  2020     +6.1%    +10.0%    +18.2%     -3.9%          83%
  2021    -15.2%     -3.3%     -4.6%    -11.9%          56%
  2022     +1.4%    -15.2%    -31.2%    

### The wins that are not timing

Look at 2011 (+10.7 over IEF) and 2014 (+15.9). Those are bigger gaps than most of the losses, and they have nothing to do with reading the front end: the rule simply happened to be holding TLT while TLT returned +34% and +27%. It **lagged its own instrument** in both years — it beat IEF only because twenty-year duration beats eight-year duration in a bond bull market. The bill for that beta arrives in 2009 (−14.9), 2015 (−12.8) and 2021 (−11.9), when it held the same duration into the sell-off.

Strip out the beta and the timing record really is 2022 and 2023 — two episodes in nineteen years.

## 4. "But it beat a coin flip!" — the trap

The standard fairness check is a **random control**: a rule that owns long bonds on *random* days, but just as often (48% of the time). If our rule beats it, the *choice of days* was worth something.

On one random draw ours wins by a convincing **+0.469** Sharpe. But a coin flipped 4,800 times is one draw of many, so we ran **30** of them — and then ran them again with the trading costs switched off:

| | advantage over random | how sure |
|---|--:|--:|
| after costs | **+0.268** | *t* = +1.20 |
| **before costs** | **+0.060** | *t* = +0.29 |

The win evaporates when costs come off. The reason is mundane: a coin flip changes its mind about **2,400** times over the sample; our rule changes its mind **321** times. We were not out-*picking* the coin, we were out-*sitting* it. That is a real advantage over a maniac, but it is not a forecast.

**The fair version of that test.** Give the rule an opponent with its exact average exposure and *no* switching at all: a fixed **47.8% TLT / 52.2% bills** portfolio, held throughout. Before costs the two are level (+0.185 vs +0.184 — a gap of **+0.001**); after costs the lazy blend is **ahead** (+0.182 vs +0.154). Nineteen years of decisions bought exactly nothing that owning the average would not have given you for free.

## 5. And it depends entirely on a number we made up

Why three months? No reason — it just sounds sensible. Try the neighbours:

| lookback | advantage over IEF |
|---|--:|
| 1 month | +0.056 |
| 2 months | -0.178 |
| **3 months** (ours) | **-0.139** |
| 6 months | -0.122 |
| 12 months | +0.083 |

The answer changes **sign three times** across a perfectly reasonable grid, and never gets big enough to matter in either direction. When the conclusion depends on an arbitrary knob, the conclusion is the knob.

## 6. Is the test itself any good? (live, offline, synthetic)

Before believing a null result you have to prove the measuring device works. So we build a **fake world** where short rates genuinely do trend — where this month's rate move really does predict next month's — and check that the rule finds it. Then we build a world where rate moves are pure coin flips and check that the rule finds nothing.

> 🔬 **For the quants** — the knob is the AR(1) coefficient on daily rate *increments*, with unconditional rate volatility held fixed, so the two worlds differ in predictability and not in risk. Nothing in this cell touches the real tape.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from front_end_trend import data, strategy as st

trending = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=925)[0])
coinflip = st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=925)[0])
print('SYNTHETIC (not the real tape)')
print(f"world where rates really trend : Sharpe advantage "
      f"{trending['excess_sharpe_adv']:+.2f}  -> the rule finds it easily")
print(f"world of pure coin-flip rates  : Sharpe advantage "
      f"{coinflip['excess_sharpe_adv']:+.2f}  -> and it correctly finds nothing")
print()
print('So the measuring device works. The blank on the real tape is the tape\'s.')

SYNTHETIC (not the real tape)
world where rates really trend : Sharpe advantage +1.78  -> the rule finds it easily
world of pure coin-flip rates  : Sharpe advantage -0.06  -> and it correctly finds nothing

So the measuring device works. The blank on the real tape is the tape's.


## Verdict

- **Signal — None.** Against simply holding IEF, the front-end trend rule is behind by **-0.139** Sharpe with a *t* of -0.20 — no signal, and the point estimate has the wrong sign. The answer flips with the lookback window, flips between the first and second halves of the sample, and the one apparently impressive result (beating a coin flip) turns out to be a turnover artefact that vanishes before costs. Against a lazy fixed-weight portfolio of the same average exposure the rule is level before costs and behind after them.
- **Tradability — Mirage.** Lower return, higher volatility, deeper drawdown, ~17 round trips a year, and it beats its benchmark in only 11 of 20 years for a mean gap of -0.10 pp/yr. You would be paying to make your bond sleeve worse.
- **What is true.** The rule really did sit out all of 2022 and really did hold duration through the late-2023 rally. Both are honest — and its other big years (2011, 2014) are long-duration beta, not timing. A nineteen-year record whose timing content is two loud episodes is the exact shape of luck, and the tape here cannot tell it from anything else.